# 🦅 Set Up

''' To use GPU
conda install -y pytorch pytorch-cuda=12.1 -c pytorch -c nvidia
'''

In [1]:
from functions import *

In [2]:
cd ..

C:\Users\chopi\Penn Dropbox\Hyunwoo Jung\1_Personal\_Hyunwoo Place\graduate school\2_coursework (2025-F)\2_CIS5200_Machine Learning\5_final project\3_analyses


# 🦅 Load Data

In [3]:
df_reviews = pd.read_feather('data/appliances_reviews_012023_062023 v1.4.0.ftr')

In [4]:
df_reviews.head()

,review_id,rating,title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,...,ResNet503,ResNet504,ResNet505,ResNet506,ResNet507,ResNet508,ResNet509,ResNet510,ResNet511,ResNet512
0,1,3.0,Needs hose clamps,Needs metal hose clamps not plastic ties.... C...,B00004YWK2,B00004YWK2,AFFPAJDCW7NSKE4FZWBRWETUKZ2A,2023-02-18 02:32:06.278,0,True,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2,1.0,Don't waste your money,Product is cheap and the door doesn't work well,B00004YWK2,B00004YWK2,AEI6B25VF65CG2HPBQ2FNBG7IQKA,2023-02-10 00:31:39.499,0,True,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,3,1.0,Must have been a return,No cover to close it and plastic was separated.,B00004YWK2,B00004YWK2,AGA5X6NUWSQM42KDFJLO25JBXOEA,2023-02-07 14:55:58.766,0,True,...,0.713876,1.146581,0.103412,0.564442,1.773487,0.983913,0.220921,1.707405,0.076411,0.557772
3,4,5.0,Great product,"I love this it keeps my garage, nice and warm ...",B00004YWK2,B00004YWK2,AGO4SBTXOUTKYMHKQQNX7ZFDQSFA,2023-01-29 19:37:40.587,0,True,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,5,5.0,Why be wasteful?,"So I mounted this, as you see, behind and just...",B00004YWK2,B00004YWK2,AHKTIX6L7FKPYDENUALEPNPMAIHQ,2023-01-03 16:32:51.697,0,True,...,1.749199,1.291329,1.209663,0.281747,0.468356,0.669414,0.217397,1.138075,0.267219,0.775042


# 🦅 SAE

## 🐔 all reviews

### 🐤 data prep

In [5]:
df = df_reviews[df_reviews['inter_review_time'].notnull()][['review_id', 'parent_asin', 'text', 'inter_review_time']].rename(columns={'parent_asin':'item_id'})

In [6]:
df['irt_days'] = df['inter_review_time'].dt.total_seconds()/(3600*24)
df['y_log_irt'] = np.log(df['irt_days']+1)

In [7]:
df[["item_id","review_id","text","irt_days","y_log_irt"]].head()

,item_id,review_id,text,irt_days,y_log_irt
1,B00004YWK2,2,Product is cheap and the door doesn't work well,8.083643,2.206475
2,B00004YWK2,3,No cover to close it and plastic was separated.,2.399777,1.223710
3,B00004YWK2,4,"I love this it keeps my garage, nice and warm ...",8.804377,2.282829
4,B00004YWK2,5,"So I mounted this, as you see, behind and just...",26.128344,3.300579
5,B00004YWK2,6,I have 3 if these and the 1st bought several y...,0.163448,0.151388


In [8]:
df[['y_log_irt']].describe().T

,count,mean,std,min,25%,50%,75%,max
y_log_irt,94297.0,1.636924,1.221443,0.0,0.594817,1.431195,2.536008,5.188951


### 🐤 freeze LM

In [61]:
# Frozen LM
lm = FrozenLM(LMConfig(model_name="roberta-base", layer_index=10))

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [45]:
df["text"].tolist()[:2]

["Product is cheap and the door doesn't work well",
 'No cover to close it and plastic was separated.']

In [44]:
lm.token_activations( df["text"].tolist()[:2])

{'input_ids': tensor([[    0, 41257,    16,  6162,     8,     5,  1883,   630,    75,   173,
           157,     2],
        [    0,  3084,  1719,     7,   593,    24,     8,  4136,    21,  8254,
             4,     2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]), 'offset_mapping': tensor([[[ 0,  0],
         [ 0,  7],
         [ 8, 10],
         [11, 16],
         [17, 20],
         [21, 24],
         [25, 29],
         [30, 35],
         [35, 37],
         [38, 42],
         [43, 47],
         [ 0,  0]],

        [[ 0,  0],
         [ 0,  2],
         [ 3,  8],
         [ 9, 11],
         [12, 17],
         [18, 20],
         [21, 24],
         [25, 32],
         [33, 36],
         [37, 46],
         [46, 47],
         [ 0,  0]]])}
X torch.Size([2, 12, 768]) <built-in method size of Tensor object at 0x0000026AAAE4A860>
tensor([[[-0.0149,  0.0247, -0.0881,  ...,  0.0823,  0.0730,  0.0202],
         [ 0.1389,  1.2086,

([tensor([[ 0.1389,  1.2086, -0.2203,  ...,  0.4944,  0.7045, -0.0818],
          [ 0.0673,  0.8294,  0.0670,  ..., -0.3470,  0.0656, -0.4067],
          [ 0.2342, -0.2738,  0.0787,  ...,  0.6736,  0.1009, -0.0113],
          ...,
          [ 0.0512,  1.1095, -0.2456,  ..., -0.8284,  0.0578,  0.0718],
          [ 0.3855,  0.6714,  0.1311,  ...,  0.7857,  0.1904,  0.3321],
          [-0.0314,  0.9862, -0.1390,  ...,  0.1123,  0.1070,  0.3505]]),
  tensor([[-0.3460,  0.3207, -0.3603,  ..., -0.0014,  0.0660,  0.2924],
          [ 0.0273, -0.0054,  0.1645,  ...,  0.4341,  0.6685, -0.0678],
          [-0.1894,  0.0049, -0.1399,  ...,  0.4066,  0.5038, -0.2618],
          ...,
          [ 0.1076,  0.8023, -0.0785,  ..., -0.2408, -0.2541,  0.1046],
          [ 0.0737,  0.4294, -0.1156,  ..., -0.2062, -0.0043, -0.2289],
          [-0.0072,  0.0437, -0.0105,  ..., -0.0056,  0.0174, -0.0073]])],
 [[41257, 16, 6162, 8, 5, 1883, 630, 75, 173, 157],
  [3084, 1719, 7, 593, 24, 8, 4136, 21, 8254, 4]]

### 🐤 train a SAE

In [47]:
# SAE training on *unlabeled* texts (ideally, use a large pool)
unlabeled_corpus = df["text"].tolist()  # replace with bigger corpus
print(len(unlabeled_corpus))
token_stream = TokenBatcher(unlabeled_corpus, lm, batch_tokens=8192)

94297


In [15]:

d_model = lm.lm.config.hidden_size
K = 4 * d_model
sae = train_sae(token_stream, d_model=d_model, k=K, lam=1e-3, steps=1000, device=lm.cfg.device)

94297
[200/1000] recon=0.1484  active/token=0.249
[400/1000] recon=0.0158  active/token=0.362
[600/1000] recon=0.0039  active/token=0.425
[800/1000] recon=0.0013  active/token=0.455
[1000/1000] recon=0.0008  active/token=0.463


### 🐤 save model

In [16]:
date = datetime.now().strftime("%m%d_%Y_%H%M%S")

save_path = "data/model/sae_robertaL{}_k{}_{}.pt".format(lm.cfg.layer_index, K, date)

ckpt = {
    "state_dict": sae.state_dict(),                 # weights
    "arch": {"d_model": int(d_model), "k": int(K)}, # shapes to rebuild the module
    "meta": {
        "base_model": "roberta-base",
        "layer_index": int(lm.cfg.layer_index),
        "lambda": 1e-3,
        "steps": 1000,
        "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    },
}
torch.save(ckpt, save_path)

# (optional) human-readable sidecar
with open(save_path + ".meta.json", "w", encoding="utf-8") as f:
    json.dump({"arch": ckpt["arch"], "meta": ckpt["meta"]}, f, indent=2)

### 🐤 get the embeddings

In [62]:
# usage
sae_loaded, meta = load_sae("data/model/sae_robertaL10_k3072_1115_2025_113219.pt", device=lm.cfg.device)

C:\Users\chopi\Penn Dropbox\Hyunwoo Jung\1_Personal\_Hyunwoo Place\graduate school\2_coursework (2025-F)\2_CIS5200_Machine Learning\5_final project\3_analyses\jupyter\functions.py:527: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitH

In [19]:
''' How to merge '''
sae_feat_df = make_sae_feature_df(lm, sae_loaded, df, text_col="text", id_col="review_id",
                                  pool="topk", topk=3, prefix="sae_")

# Merge features back to the original rows
df_with_sae = df.merge(sae_feat_df, on="review_id", how="left")
print(df_with_sae.shape)
df_with_sae.head()

SAE->review features: 100%|████████████████████████████████████████████████████████| 2947/2947 [06:17<00:00,  7.81it/s]


(94297, 3078)


,review_id,item_id,text,inter_review_time,irt_days,y_log_irt,sae_0,sae_1,sae_2,sae_3,...,sae_3062,sae_3063,sae_3064,sae_3065,sae_3066,sae_3067,sae_3068,sae_3069,sae_3070,sae_3071
0,2,B00004YWK2,Product is cheap and the door doesn't work well,8 days 02:00:26.779000,8.083643,2.206475,0.000000,1.425354,1.305985,0.000000,...,0.000000,0.874277,0.0,0.000000,0.959373,0.724116,0.324260,0.478404,1.088189,1.041039
1,3,B00004YWK2,No cover to close it and plastic was separated.,2 days 09:35:40.733000,2.399777,1.223710,0.224967,1.231542,1.122654,0.101840,...,0.085051,0.900687,0.0,0.000000,0.844423,1.076546,0.572009,0.446828,0.940490,0.996120
2,4,B00004YWK2,"I love this it keeps my garage, nice and warm ...",8 days 19:18:18.179000,8.804377,2.282829,0.226200,1.067915,1.358308,0.102711,...,0.085435,1.028622,0.0,0.000000,1.070427,1.031734,0.867664,0.799850,0.789101,1.055278
3,5,B00004YWK2,"So I mounted this, as you see, behind and just...",26 days 03:04:48.890000,26.128344,3.300579,0.663035,1.612317,1.635834,0.299319,...,0.244701,1.255791,0.0,0.020767,1.479797,1.525874,0.999281,0.714161,2.252168,1.387975
4,6,B00004YWK2,I have 3 if these and the 1st bought several y...,0 days 03:55:21.873000,0.163448,0.151388,0.477116,1.353411,1.241897,0.195181,...,0.167203,1.241352,0.0,0.000000,1.216864,0.925853,0.965894,0.587046,1.186975,1.142432


In [20]:
df_with_sae.to_feather('data/appliances_reviews_012023_062023 v1.4.0 (SAE embedding).ftr')

### 🐤 predict with RandomForest

In [63]:
df_with_sae = pd.read_feather('data/appliances_reviews_012023_062023 v1.4.0 (SAE embedding).ftr')

In [64]:
# Split (time-aware split is recommended)
m = int(0.7 * len(df_with_sae))
df_tr, df_va = df_with_sae.iloc[:m], df_with_sae.iloc[m:]

In [65]:
# Build review-level features and fit downstream model
F_tr = df_tr.drop(columns=['review_id', 'item_id', 'text', 'inter_review_time', 'irt_days', 'y_log_irt']).values
F_va = df_va.drop(columns=['review_id', 'item_id', 'text', 'inter_review_time', 'irt_days', 'y_log_irt']).values
texts_va = df_va['text'].values

In [67]:
model, metrics = fit_downstream_RF(F_tr, df_tr["y_log_irt"].values, F_va, df_va["y_log_irt"].values)
print(metrics)

{'R2': -0.0445678550961075, 'RMSE': 1.1396785278822599}


In [68]:
# 1) SHAP values on validation set
explainer = shap.TreeExplainer(model)
sv = explainer.shap_values(F_va)                 # shape: [n_samples, n_features]
mean_abs = np.abs(sv).mean(axis=0)                  # magnitude (importance)
mean_signed = sv.mean(axis=0)                       # direction on average (±)

order = np.argsort(mean_abs)[::-1]
topK = min(20, len(order))
sae_cols = [f"sae_f{i}" for i in range(model.n_features_in_)]

print("Top features by |SHAP| (validation):")
for idx in order[:topK]:
    direction = "↑ IRT (lower demand)" if mean_signed[idx] > 0 else (
                "↓ IRT (higher demand)" if mean_signed[idx] < 0 else "neutral")
    print(f"{idx:>5}  {sae_cols[idx]:<40}  |shap|={mean_abs[idx]:.5f}  mean_shap={mean_signed[idx]:+.5f}  {direction}")

Top features by |SHAP| (validation):
 1571  sae_f1571                                 |shap|=0.06269  mean_shap=-0.02138  ↓ IRT (higher demand)
 2852  sae_f2852                                 |shap|=0.03350  mean_shap=-0.01353  ↓ IRT (higher demand)
  204  sae_f204                                  |shap|=0.02672  mean_shap=+0.00143  ↑ IRT (lower demand)
  929  sae_f929                                  |shap|=0.02521  mean_shap=-0.00249  ↓ IRT (higher demand)
 2217  sae_f2217                                 |shap|=0.02377  mean_shap=-0.01025  ↓ IRT (higher demand)
 1234  sae_f1234                                 |shap|=0.01891  mean_shap=+0.00232  ↑ IRT (lower demand)
  942  sae_f942                                  |shap|=0.01260  mean_shap=-0.00612  ↓ IRT (higher demand)
 1891  sae_f1891                                 |shap|=0.01042  mean_shap=-0.00209  ↓ IRT (higher demand)
 1029  sae_f1029                                 |shap|=0.00941  mean_shap=-0.00032  ↓ IRT (higher demand)
 1

In [28]:
inspect_sae_feature(feature_idx=204, lm=lm, sae=sae, F_va=F_va, texts_va=texts_va, order=order, n_reviews_to_show=3)


Inspecting SAE Feature: 204

--- Top Example 1 (Review Index: 10545) ---
Review-Level Feature Activation: 2.0952

Highlighted Review (Feature Activation Points in RED):
If you have an espresso machine, you know how important it is to keep the machine clean inside and out.<br />This little screen is so simple, but it saves a lot of work.<br />I just drop the screen on top of my prepared puck and make my espresso as usual.<br />The screen prevents coffee from contacting the shower screen in my E61 group head.  This means that coffee isn't getting up into the group drain valve or the over-pressure valve.  ... and that means that these two valves won't get gummed up with coffee.  It also means that the shower head isn't collecting residual coffee.  In short,  the insides of my espresso machine remain clean.<br /><br />Does this little puck screen make better coffee?  Maybe.  Probably more consistent.  But to be honest my puck prep is pretty good and I don't get a lot of channeling.  But I